In [1]:
import os
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

os.environ["WANDB_DISABLED"] = "true"

MODEL_NAME = "google/flan-t5-large"
OUTPUT_DIR = "outputs/flan-t5-large-idiom-combined"

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: NVIDIA GeForce RTX 3060 Ti


In [8]:
import csv
import os
import random
import re
from pathlib import Path
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

SEED = 42
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128
TRAIN_EPOCHS = 20
LEARNING_RATE = 1e-4
BATCH_SIZE = 2

# Keep this small for quick smoke tests. Set to None to train on every CSV row.
MAX_REAL_ROWS = None
INCLUDE_GENERATION_TASK = True
TEST_FRACTION = 0.2
SAMPLE_EXAMPLES_PER_TASK = 2
SANITY_CHECK_EXAMPLES = 2

set_seed(SEED)

# Load the saved fine-tuned model
# Use OUTPUT_DIR so it loads the trained checkpoint instead of the base model.
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(device)
print(f"Loaded saved model from {OUTPUT_DIR}")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded saved model from outputs/flan-t5-large-idiom-combined


In [3]:
# data loading

CSV_CANDIDATES = [
    Path("dataset/idioms_dataset.csv"),
    # Path("data/flute_idioms.csv"),
    # Path("flute_idioms.csv"),
]

data_csv = next((path for path in CSV_CANDIDATES if path.exists()), None)


def mask_idiom(text, idiom, mask="[IDIOM]"):
    escaped_idiom = re.escape(idiom.strip())
    return re.sub(rf"(?<!\w){escaped_idiom}(?!\w)", mask, text, flags=re.IGNORECASE)


def make_interpretation_example(row):
    idiom = row["idiom"].strip()
    sentence = row["example"].strip()
    meaning = row["explanation_correct"].strip()
    prompt = (
        f"Idiom: {idiom}\n"
        f"Sentence: {sentence}"
    )
    return {
        "task": "interpretation",
        "input": prompt,
        "target": meaning,
    }


def make_generation_example(row):
    idiom = row["idiom"].strip()
    sentence = row["example"].strip()
    meaning = row["explanation_correct"].strip()
    scenario = row.get("correct_substitution", "").strip() or sentence
    masked_meaning = mask_idiom(meaning, idiom)
    masked_scenario = mask_idiom(scenario, idiom)
    prompt = (
        f"Meaning: {masked_meaning}\n"
        f"Scenario: {masked_scenario}"
    )
    return {
        "task": "generation",
        "input": prompt,
        "target": f"Idiom: {idiom}\nSentence: {sentence}",
    }


if data_csv is None:
    expected_paths = ", ".join(str(path) for path in CSV_CANDIDATES)
    raise FileNotFoundError(f"Could not find formatted FLUTE CSV. Expected one of: {expected_paths}")

with data_csv.open("r", encoding="utf-8-sig", newline="") as csv_file:
    rows = list(csv.DictReader(csv_file))

required_columns = {"idiom", "example", "explanation_correct"}
missing_columns = required_columns - set(rows[0].keys() if rows else [])
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {sorted(missing_columns)}")

rows = [row for row in rows if all(row.get(column, "").strip() for column in required_columns)]
if MAX_REAL_ROWS is not None:
    rows = rows[:MAX_REAL_ROWS]

if len(rows) < 2:
    raise ValueError("Need at least 2 usable CSV rows to make a train/test split.")

split_rows = rows.copy()
random.Random(SEED).shuffle(split_rows)

test_row_count = max(1, round(len(split_rows) * TEST_FRACTION))
test_row_count = min(test_row_count, len(split_rows) - 1)
test_rows = split_rows[:test_row_count]
train_rows = split_rows[test_row_count:]


def build_task_examples(source_rows):
    examples = []
    for row in source_rows:
        examples.append(make_interpretation_example(row))
        if INCLUDE_GENERATION_TASK:
            examples.append(make_generation_example(row))
    return examples


train_examples = build_task_examples(train_rows)
test_examples = build_task_examples(test_rows)

print(f"Loaded {len(rows)} usable CSV rows from {data_csv}")
print(f"Split into {len(train_rows)} training rows and {len(test_rows)} held-out test rows")
print(f"Built {len(train_examples)} training examples and {len(test_examples)} held-out test examples")

train_dataset = Dataset.from_list(train_examples)
test_dataset = Dataset.from_list(test_examples)
test_interpretation_examples = [example for example in test_examples if example["task"] == "interpretation"]
test_generation_examples = [example for example in test_examples if example["task"] == "generation"]

preview_interpretation_examples = test_interpretation_examples[:SAMPLE_EXAMPLES_PER_TASK]
preview_generation_examples = test_generation_examples[:SAMPLE_EXAMPLES_PER_TASK]
train_dataset

Loaded 1515 usable CSV rows from dataset\idioms_dataset.csv
Split into 1212 training rows and 303 held-out test rows
Built 2424 training examples and 606 held-out test examples


Dataset({
    features: ['task', 'input', 'target'],
    num_rows: 2424
})

In [4]:
# sampling

def generate_text(prompt, max_new_tokens=96, do_sample=False, temperature=0.7):
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(model.device)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "num_beams": 4,
        "do_sample": do_sample,
    }
    if do_sample:
        generation_kwargs["temperature"] = temperature
        generation_kwargs["top_p"] = 0.9

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(**encoded, **generation_kwargs)

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def sample_examples(title, examples):
    print("=" * 80)
    print(title)
    print("=" * 80)
    if not examples:
        print("No examples available for this task.")
        return
    for idx, example in enumerate(examples, start=1):
        print(f"\nExample {idx}: {example['task']}")
        print("-" * 80)
        print("PROMPT:")
        print(example["input"])
        print("\nMODEL OUTPUT:")
        print(generate_text(example["input"]))
        print("\nTARGET:")
        print(example["target"])


sample_examples("Testing: interpretation", preview_interpretation_examples)

Testing: interpretation

Example 1: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: in your face
Sentence: J is angry, but he realises you haven't got to get totally in-your-face to get a serious message across.

MODEL OUTPUT:
To get in your face means to act with resentment, but in this context they are realising that you don't have to act like a bully to get a serious message across.

TARGET:
In your face means brashly confrontational, which J recognises is unnecessary.

Example 2: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: same difference
Sentence: With a beard, without a beard, same difference.

MODEL OUTPUT:
The idiom same difference means that two things are the same, so in this context it is saying that if one has a beard and another has no beard, it is the same as if one had a beard.

TARGET:
Same difference means the distinction makes no practical 

In [5]:
# sample generation examples after fine-tuning
sample_examples("Testing: generation", preview_generation_examples)

Testing: generation

Example 1: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: [IDIOM] means brashly confrontational, which J recognises is unnecessary.
Scenario: J is angry, but he realises you haven't got to get totally aggressive and confrontational to get a serious message across.

MODEL OUTPUT:
Idiom: on the hot seat Sentence: J is angry, but he realises you haven't got to get on the hot seat to get a serious message across.

TARGET:
Idiom: in your face
Sentence: J is angry, but he realises you haven't got to get totally in-your-face to get a serious message across.

Example 2: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: [IDIOM] means the distinction makes no practical difference.
Scenario: With a beard, without a beard, it amounts to the same thing.

MODEL OUTPUT:
Idiom: no difference Sentence: With a beard, without a beard, it matters no matter what.

TA

In [6]:
# Evaluate the fine-tuned model with BERTScore on the held-out test set.
# Compare each model prediction against that same example's target text.
from bert_score import score

predictions = []
references = []

for example in test_interpretation_examples:
    prompt = example["input"]
    target = example["target"]

    pred_text = generate_text(prompt)

    predictions.append(pred_text)
    references.append(target)

# Compute BERTScore across the whole test set
P, R, F1 = score(
    predictions, 
    references, 
    model_type=model.config._name_or_path,
    num_layers=24,
    # device=device
    )

bertscore_summary = {
    "precision": float(P.mean().item()),
    "recall": float(R.mean().item()),
    "f1": float(F1.mean().item()),
}

print(f"BERTScore on test set:")
print(f"  Precision: {bertscore_summary['precision']:.4f}")
print(f"  Recall:    {bertscore_summary['recall']:.4f}")
print(f"  F1:        {bertscore_summary['f1']:.4f}")

# Optional: show a few example predictions
print("\nSample predictions vs targets:")
for i in range(min(3, len(predictions))):
    print(f"\nExample {i + 1}")
    print("INPUT:   ", test_interpretation_examples[i]["input"])
    print("PRED:    ", predictions[i])
    print("TARGET:  ", references[i])

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

[transformers] T5EncoderModel LOAD REPORT from: outputs/flan-t5-large-idiom-combined
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore on test set:
  Precision: 0.6363
  Recall:    0.6458
  F1:        0.6378

Sample predictions vs targets:

Example 1
INPUT:    Idiom: in your face
Sentence: J is angry, but he realises you haven't got to get totally in-your-face to get a serious message across.
PRED:     To get in your face means to act with resentment, but in this context they are realising that you don't have to act like a bully to get a serious message across.
TARGET:   In your face means brashly confrontational, which J recognises is unnecessary.

Example 2
INPUT:    Idiom: same difference
Sentence: With a beard, without a beard, same difference.
PRED:     The idiom same difference means that two things are the same, so in this context it is saying that if one has a beard and another has no beard, it is the same as if one had a beard.
TARGET:   Same difference means the distinction makes no practical difference.

Example 3
INPUT:    Idiom: turn over a new leaf
Sentence: There is no indication that Hollywoo

In [ ]:
# Load custom_generation.csv
custom_gen_csv = pd.read_csv('dataset/custom_generation.csv')

def normalize_quotes(text):
    """Replace curly quotes with ASCII apostrophes and quotes"""
    text = text.replace('\u2018', "'") 
    text = text.replace('\u2019', "'") 
    text = text.replace('\u201C', '"')
    text = text.replace('\u201D', '"') 
    return text

results = []

for idx, row in custom_gen_csv.iterrows():
    task = row['task']
    input_text = row['input']
    
    prompt = f"{task.lower()}: {input_text}"
    
    prediction = generate_text(prompt, max_new_tokens=128)
    
    prediction = normalize_quotes(prediction)
    normalized_input = normalize_quotes(input_text)
    
    results.append({
        'task': task,
        'input': normalized_input,
        'prediction': prediction
    })
    
    print(f"Example {idx + 1}/{len(custom_gen_csv)}:")
    print(f"Input: {input_text[:150]}...")
    print(f"Output: {prediction}\n")

print(f"\n Inference complete! Processed {len(results)} examples.")

Example 1/46:
Input: Meaning: The saying [IDIOM] means to perform flawlessly, such as in a performance or implementation. Scenario: That player performed so well last nigh...
Output: Idiom: out of this world Sentence: That player out of this world last night in their victory against the opposing team.

Example 2/46:
Input: Meaning: [IDIOM] is a saying that means with much regret. Scenario: Kelvin felt very regretful when he told Sarah the news of their stock prices falli...
Output: Idiom: with a heavy heart Sentence: Kelvin felt with a heavy heart when he told Sarah the news of their stock prices falling.

Example 3/46:
Input: Meaning: The idiom [IDIOM] means to be very hungry. Scenario: Wow, I haven’t eaten in 10 hours, I am so hungry right now....
Output: Idiom: on a full stomach Sentence: Wow, I haven't eaten in 10 hours, I am on a full stomach right now.

Example 4/46:
Input: Meaning: The idiom [IDIOM] means to be super plush and soft in a luscious and luxurious manner. Scenario: A

In [ ]:
# Save results to CSV
import csv

results_df = pd.DataFrame(results)
output_path = 'custom_generation_results.csv'

results_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"Results saved to {output_path}")

Results saved to custom_generation_results.csv
